<a href="https://colab.research.google.com/github/sayandxzzz/sayan/blob/main/fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nltk scikit-learn gradio

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pickle

nltk.download('stopwords')

from nltk.corpus import stopwords

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
pf_real = pd.read_csv("PolitiFact_real_news_content.csv")
pf_fake = pd.read_csv("PolitiFact_fake_news_content.csv")

bf_real = pd.read_csv("BuzzFeed_real_news_content.csv")
bf_fake = pd.read_csv("BuzzFeed_fake_news_content.csv")

In [ ]:
pf_real["label"] = 1
bf_real["label"] = 1

pf_fake["label"] = 0
bf_fake["label"] = 0

In [ ]:
data = pd.concat(
    [pf_real, bf_real, pf_fake, bf_fake],
    ignore_index=True
)

data = data[['title','text','label']]

print(data.shape)

data.head()

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r'http\\S+','',text)

    text = re.sub(r'[^a-zA-Z ]',' ',text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    return " ".join(words)

data['content'] = data['title'].fillna('') + " " + data['text'].fillna('')

data['content'] = data['content'].apply(clean_text)

data.head()

In [ ]:
X = data['content']

y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train = vectorizer.fit_transform(X_train)

X_test = vectorizer.transform(X_test)

In [ ]:
model = LogisticRegression()

model.fit(
    X_train,
    y_train
)

In [ ]:
pred = model.predict(X_test)

acc = accuracy_score(
    y_test,
    pred
)

print(
    "Accuracy:",
    acc
)

print(
    classification_report(
        y_test,
        pred
    )
)

print(
    confusion_matrix(
        y_test,
        pred
    )
)

In [ ]:
pickle.dump(
    model,
    open(
        "fake_news_model.pkl",
        "wb"
    )
)

pickle.dump(
    vectorizer,
    open(
        "vectorizer.pkl",
        "wb"
    )
)

print("Saved")

In [ ]:
import gradio as gr

def detect_news(news):

    news = clean_text(news)

    vec = vectorizer.transform([news])

    pred = model.predict(vec)

    if pred[0] == 1:
        return "REAL NEWS"

    else:
        return "FAKE NEWS"

app = gr.Interface(
    fn=detect_news,
    inputs="textbox",
    outputs="text",
    title="Fake News Detector"
)

app.launch()